# BERT (Bidirectional Encoder Representations from Transformers) - Mathematical Aspects

BERT introduced several groundbreaking innovations in natural language processing. Here's a deep dive into the mathematical aspects of each key innovation:

## 1. Bidirectional Contextualization

### Mathematical Details:
- **Bidirectional Attention:** BERT uses the Transformer encoder, applying self-attention in both directions (left-to-right and right-to-left).

  - **Attention Score Calculation:**
    $\text{Attention}(Q, K, V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}}) V$
    
    where:
    - $Q$ (Query) is derived from the current token's representation,
    - $K$ (Key) and $V$ (Value) are derived from all tokens in the sequence,
    - $d_k$ is the dimension of the key vectors.

  - **Bidirectional Context:** Each token's representation incorporates both left and right context.

  - **Mathematical Representation:** The output representation for token $w_i$ is:
  $h_i = \text{LayerNorm}(\text{Attention}(Q_i, K, V) + h_i^{\text{previous}})$

    where $h_i^{\text{previous}}$ is the previous layer's output for $w_i$.

### Impact:
- **Enhanced Context Understanding:** Bidirectional attention leads to a more comprehensive understanding of token meanings.

## 2. Masked Language Model (MLM)

### Mathematical Details:
- **Masking Mechanism:** During training, BERT masks a proportion of tokens and trains the model to predict these masked tokens.

  - **Loss Function:** For a masked token $w_i$, the loss function is cross-entropy:

    $L_{\text{MLM}} = -\sum_{i} \log p(w_i | \text{context})$
       
    where $p(w_i | \text{context})$ is the predicted probability of the masked token.

  - **Training Objective:** Minimize the negative log likelihood of predicting masked tokens.

### Impact:
- **Contextual Representation Learning:** Enables learning of context-dependent word representations.

## 3. Next Sentence Prediction (NSP)

### Mathematical Details:
- **Sentence Pair Classification:** NSP involves predicting whether one sentence follows another.

  - **Loss Function:** For sentence pairs, the loss function is binary cross-entropy:
    
    $L_{\text{NSP}} = -[ y \log p(\text{is\_next}) + (1 - y) \log (1 - p(\text{is\_next})) ]$
    
    where $y$ is the binary label and $p(\text{is\_next})$ is the probability of the second sentence following the first.

  - **Training Objective:** Maximize the likelihood of predicting sentence relationships correctly.

### Impact:
- **Sentence-Level Understanding:** Helps in capturing relationships between sentences.

## 4. Pre-training on Large Text Corpora

### Mathematical Details:
- **Large-Scale Training:** BERT is pretrained on datasets like BooksCorpus and English Wikipedia.

  - **Pre-training Objective:** Combines MLM and NSP objectives, optimized using stochastic gradient descent or variants like Adam:
    
    $L_{\text{total}} = L_{\text{MLM}} + \lambda L_{\text{NSP}}$
    
    where $\lambda$ is a weighting factor.

  - **Optimization:** Uses gradient-based methods to update model parameters.

### Impact:
- **Generalization:** Provides robust representations that perform well across various NLP tasks.

## 5. Fine-Tuning for Specific Tasks

### Mathematical Details:
- **Task-Specific Adaptation:** BERT is fine-tuned by adding task-specific output layers.

  - **Fine-Tuning Process:** Involves training on task-specific data, using a loss function appropriate for the task:
    
    $L_{\text{task}} = -\sum_{i} \log p(y_i | \text{features})$
    
    where $y_i$ are the labels and $\text{features}$ are the input representations.

### Impact:
- **Versatility:** Allows adaptation to various NLP tasks with minimal additional training.

## References

1. Devlin, J., Chang, M. W., Lee, K., & Toutanova, K. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *arXiv preprint arXiv:1810.04805*. [Link](https://arxiv.org/abs/1810.04805)
2. Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., Polosukhin, I. (2017). Attention is All You Need. *Advances in Neural Information Processing Systems (NeurIPS)*. [Link](https://arxiv.org/abs/1706.03762)

In [ ]:
!pip install torch==2.0.1 torchvision torchaudio

INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.0/21.0 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 44.9 MB/s eta 0:00:00

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(2)
            mask = mask.expand(-1, self.num_heads, -1, -1)
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output

    def forward(self, x, mask=None):
        batch_size = x.size(0)

        Q = self.W_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.W_o(attn_output)

class PositionwiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionwiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        attn_output = self.self_attn(x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

class BERT(nn.Module):
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout):
        super(BERT, self).__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_seq_length, d_model)

        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(d_model)

        self.mlm_head = nn.Linear(d_model, vocab_size)
        self.nsp_head = nn.Linear(d_model, 2)

        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length

        print(f"Token embedding size: {self.token_embedding.weight.shape}")
        print(f"Position embedding size: {self.position_embedding.weight.shape}")

    def forward(self, input_ids, attention_mask):
        seq_length = input_ids.size(1)
        position_ids = torch.arange(seq_length, dtype=torch.long, device=input_ids.device).unsqueeze(0).expand_as(input_ids)

        print(f"Input ids shape: {input_ids.shape}, max value: {input_ids.max()}")
        print(f"Position ids shape: {position_ids.shape}, max value: {position_ids.max()}")
        print(f"Vocab size: {self.vocab_size}")
        print(f"Max seq length: {self.max_seq_length}")

        assert input_ids.max() < self.vocab_size, f"Input ids max value {input_ids.max()} is >= vocab_size {self.vocab_size}"
        assert seq_length <= self.max_seq_length, f"Sequence length {seq_length} is > max_seq_length {self.max_seq_length}"

        token_embeddings = self.token_embedding(input_ids)
        position_embeddings = self.position_embedding(position_ids)

        x = self.dropout(token_embeddings + position_embeddings)

        for layer in self.layers:
            x = layer(x, attention_mask)

        x = self.norm(x)
        mlm_output = self.mlm_head(x)

        cls_output = x[:, 0]
        nsp_output = self.nsp_head(cls_output)

        return mlm_output, nsp_output

# Model parameters
vocab_size = 30000
d_model = 768
num_layers = 12
num_heads = 12
d_ff = 3072
max_seq_length = 512
dropout = 0.1

# Create BERT model
bert = BERT(vocab_size, d_model, num_layers, num_heads, d_ff, max_seq_length, dropout)

# Simulating input
batch_size = 32
seq_length = 128
input_ids = torch.randint(0, vocab_size - 1, (batch_size, seq_length))  # Note the -1 here
attention_mask = torch.ones_like(input_ids)
next_sentence_label = torch.randint(0, 2, (batch_size,))

# Forward pass
try:
    mlm_output, nsp_output = bert(input_ids, attention_mask)
    print("Forward pass successful!")

    # Calculating NSP loss
    nsp_loss_fn = nn.CrossEntropyLoss()
    nsp_loss = nsp_loss_fn(nsp_output, next_sentence_label)

    print(f"MLM output shape: {mlm_output.shape}")
    print(f"NSP output shape: {nsp_output.shape}")
    print(f"NSP Loss: {nsp_loss.item()}")

    # Simulating MLM training
    mlm_labels = torch.randint(0, vocab_size, (batch_size, seq_length))
    mlm_labels[torch.rand_like(mlm_labels.float()) < 0.85] = -100  # 15% masking rate

    criterion = nn.CrossEntropyLoss(ignore_index=-100)
    optimizer = optim.Adam(bert.parameters(), lr=1e-4)

    # One training step
    def train_step(model, input_ids, attention_mask, mlm_labels, nsp_labels):
        optimizer.zero_grad()
        mlm_output, nsp_output = model(input_ids, attention_mask)
        mlm_loss = criterion(mlm_output.view(-1, vocab_size), mlm_labels.view(-1))
        nsp_loss = nsp_loss_fn(nsp_output, nsp_labels)
        total_loss = mlm_loss + nsp_loss
        total_loss.backward()
        optimizer.step()
        return total_loss.item(), mlm_loss.item(), nsp_loss.item()

    # Run a training step
    total_loss, mlm_loss, nsp_loss = train_step(bert, input_ids, attention_mask, mlm_labels, next_sentence_label)

    print(f"Total Loss: {total_loss}")
    print(f"MLM Loss: {mlm_loss}")
    print(f"NSP Loss: {nsp_loss}")

except Exception as e:
    print(f"An error occurred: {str(e)}")
    import traceback
    traceback.print_exc()

Token embedding size: torch.Size([30000, 768])
Position embedding size: torch.Size([512, 768])
Input ids shape: torch.Size([32, 128]), max value: 29991
Position ids shape: torch.Size([32, 128]), max value: 127
Vocab size: 30000
Max seq length: 512
Forward pass successful!
MLM output shape: torch.Size([32, 128, 30000])
NSP output shape: torch.Size([32, 2])
NSP Loss: 0.7872638702392578
Input ids shape: torch.Size([32, 128]), max value: 29991
Position ids shape: torch.Size([32, 128]), max value: 127
Vocab size: 30000
Max seq length: 512
Total Loss: 11.407297134399414
MLM Loss: 10.459787368774414
NSP Loss: 0.9475102424621582


# BERT Implementation Overview

This code implements a BERT (Bidirectional Encoder Representations from Transformers) model. It defines the key components of BERT and sets up a basic training loop.

## Key Components

1. **MultiHeadAttention**: Implements the multi-head attention mechanism.
2. **PositionwiseFeedForward**: Defines the feed-forward network used in each transformer layer.
3. **EncoderLayer**: Combines multi-head attention and feed-forward network to form a single encoder layer.
4. **BERT**: The main model class, incorporating all components.

## BERT Class Details

- Token and position embeddings
- Multiple encoder layers
- Masked Language Model (MLM) head
- Next Sentence Prediction (NSP) head

## Model Parameters

- Vocabulary size: 30,000
- Model dimension: 768
- Number of layers: 12
- Number of attention heads: 12
- Feed-forward dimension: 3072
- Maximum sequence length: 512

## Simulation and Training

1. Creates a BERT model instance
2. Simulates a forward pass with random input data
3. Includes a training step function that:
   - Computes MLM and NSP losses
   - Performs backpropagation

This implementation captures the core architecture and training objectives of BERT. It's a simplified version compared to full-scale models but is suitable for understanding BERT basics or for small-scale experiments.